In [ ]:
!pip install mlflow

In [ ]:
import mlflow
from ultralytics import YOLO
import warnings

warnings.filterwarnings("ignore")

mlflow.set_experiment("Comparacao_Tamanhos_YOLO_Wafer")

modelos_para_testar = ['yolo26s-seg.pt', 'yolo26m-seg.pt','yolo26l-seg.pt']

for nome_modelo in modelos_para_testar:

    print(f"\n{'='*50}")
    print(f" INICIANDO TREINO DO MODELO: {nome_modelo}")
    print(f"{'='*50}\n")

    #carrega a arquitetura da vez
    model = YOLO(nome_modelo)

    #sub-pasta no MLflow com o nome exato do modelo
    with mlflow.start_run(run_name=f"Treino_{nome_modelo.split('.')[0]}_Aug"):
        results = model.train(
            data='./data.yaml',
            epochs=100,
            imgsz=768,
            batch=32,
            device='cuda',

            #Augmentations
            degrees=90.0,
            flipud=0.5,
            fliplr=0.5,
            mixup=0.15,
            copy_paste=0.3,
            erasing=0.4,

            # parametros
            workers=4,
            patience=40,

            cls=0.5,

            retina_masks=True,
            cos_lr=True,

            # Pastas locais
            project='experimentos_wafer',
            name=f"treino_{nome_modelo.split('.')[0]}_aug"
        )

    print(f"Treino do {nome_modelo} finalizado e salvo no MLflow")

In [ ]:
import pandas as pd
from ultralytics import YOLO
import os

caminhos_modelos = [
    'experimentos_wafer/treino_yolo26s-seg_aug/weights/best.pt',
    'experimentos_wafer/treino_yolo26m-seg_aug/weights/best.pt',
    'experimentos_wafer/treino_yolo26l-seg_aug/weights/best.pt'
]

comparativo_resultados = []

for path in caminhos_modelos:
    if os.path.exists(path):
        nome_exibicao = path.split('/')[1]
        print(f"Avaliando: {nome_exibicao}...")

        model = YOLO(path)

        metrics = model.val(data='./data.yaml', split='test')

        comparativo_resultados.append({
            'Experimento': nome_exibicao,
            'mAP50-95(B)': metrics.box.map,
            'mAP50(B)': metrics.box.map50,
            'Precision(B)': metrics.box.p,
            'Recall(B)': metrics.box.r,
            'mAP50-95(M)': metrics.seg.map,
            'mAP50(M)': metrics.seg.map50
        })
    else:
        print(f"Aviso: Modelo não encontrado em {path}")

# Criar DataFrame para comparação visual
df_comparacao = pd.DataFrame(comparativo_resultados)
if not df_comparacao.empty:
    display(df_comparacao.sort_values(by='mAP50-95(B)', ascending=False))
else:
    print("Nenhum modelo foi carregado com sucesso. Verifique se os caminhos das pastas estão corretos.")